# Keras Layers: Practical, Edge Cases, and Enterprise System Design

This notebook gives a practical guide to commonly used Keras layers:
1. Layer list with use case, do and do not guidance
2. Functional API examples and when to use each layer
3. Edge case examples that fail in real projects
4. Enterprise examples with system design patterns

In [1]:
# Import NumPy for sample arrays and numeric utilities.
import numpy as np
# Import TensorFlow runtime.
import tensorflow as tf
# Import top-level Keras API from TensorFlow.
from tensorflow import keras
# Import the layers namespace to access all Keras layer classes.
from tensorflow.keras import layers

# Print TensorFlow version so notebook runs are reproducible across environments.
print('TensorFlow version:', tf.__version__)
# Set TensorFlow random seed for deterministic-ish demos.
tf.random.set_seed(42)
# Set NumPy random seed for deterministic sample data generation.
np.random.seed(42)

TensorFlow version: 2.22.0-dev0+selfbuilt


## 1) Keras Layers Quick Reference (Use Case + Do and Do Not)

In [2]:
# Create a compact reference list for the most-used production layers.
# Each dictionary row stores one layer's practical guidance.
layer_reference = [
    {
        'Layer': 'Input',
        'Use Case': 'Define model contract and shapes',
        'Do': 'Name inputs for serving and monitoring',
        'Do Not': 'Rely on implicit shape inference in production'
    },
    {
        'Layer': 'Embedding',
        'Use Case': 'Sparse ids to dense vectors',
        'Do': 'Reserve index 0 for padding if masking',
        'Do Not': 'Use huge vocab without capping and OOV handling'
    },
    {
        'Layer': 'Dense',
        'Use Case': 'General nonlinear transformation',
        'Do': 'Use for tabular MLP heads and projection blocks',
        'Do Not': 'Use as first choice for spatial data'
    },
    {
        'Layer': 'Flatten',
        'Use Case': 'Convert multi-dimensional tensor to 1D vector per sample',
        'Do': 'Use before Dense when you intentionally need full feature vectorization',
        'Do Not': 'Flatten very large feature maps without checking parameter explosion'
    },
    {
        'Layer': 'Conv2D',
        'Use Case': 'Local spatial pattern extraction',
        'Do': 'Use with normalization and pooling',
        'Do Not': 'Flatten large feature maps too early'
    },
    {
        'Layer': 'MaxPooling2D',
        'Use Case': 'Downsample feature maps',
        'Do': 'Reduce compute and increase receptive field',
        'Do Not': 'Over-pool tiny inputs'
    },
    {
        'Layer': 'BatchNormalization',
        'Use Case': 'Stabilize and speed up training',
        'Do': 'Keep train and inference behavior clear',
        'Do Not': 'Freeze incorrectly during fine tuning'
    },
    {
        'Layer': 'Dropout',
        'Use Case': 'Regularization to reduce overfitting',
        'Do': 'Use moderate rates and validate impact',
        'Do Not': 'Expect effect during inference'
    },
    {
        'Layer': 'LSTM or GRU',
        'Use Case': 'Sequential dependency modeling',
        'Do': 'Use masking for variable length sequences',
        'Do Not': 'Use very long sequences without profiling latency'
    },
    {
        'Layer': 'MultiHeadAttention',
        'Use Case': 'Token to token context mixing',
        'Do': 'Use attention mask where needed',
        'Do Not': 'Ignore quadratic cost in sequence length'
    },
    {
        'Layer': 'LayerNormalization',
        'Use Case': 'Transformer style normalization',
        'Do': 'Pair with residual blocks',
        'Do Not': 'Confuse with batch dependent normalization'
    },
    {
        'Layer': 'Add or Concatenate',
        'Use Case': 'Merge branches in multi input architectures',
        'Do': 'Check shape compatibility',
        'Do Not': 'Merge tensors with mismatched semantic meaning'
    },
    {
        'Layer': 'GlobalAveragePooling2D',
        'Use Case': 'Compact spatial features for classification',
        'Do': 'Prefer over large flatten when possible',
        'Do Not': 'Use if location specific output is required'
    }
]

# Print each row in a readable console format.
for row in layer_reference:
    # Print layer name.
    print(f"{row['Layer']}:")
    # Print recommended use case.
    print(f"  Use Case: {row['Use Case']}")
    # Print recommended practice.
    print(f"  Do: {row['Do']}")
    # Print anti-pattern to avoid.
    print(f"  Do Not: {row['Do Not']}")
    # Print a blank line between entries.
    print()

Input:
  Use Case: Define model contract and shapes
  Do: Name inputs for serving and monitoring
  Do Not: Rely on implicit shape inference in production

Embedding:
  Use Case: Sparse ids to dense vectors
  Do: Reserve index 0 for padding if masking
  Do Not: Use huge vocab without capping and OOV handling

Dense:
  Use Case: General nonlinear transformation
  Do: Use for tabular MLP heads and projection blocks
  Do Not: Use as first choice for spatial data

Flatten:
  Use Case: Convert multi-dimensional tensor to 1D vector per sample
  Do: Use before Dense when you intentionally need full feature vectorization
  Do Not: Flatten very large feature maps without checking parameter explosion

Conv2D:
  Use Case: Local spatial pattern extraction
  Do: Use with normalization and pooling
  Do Not: Flatten large feature maps too early

MaxPooling2D:
  Use Case: Downsample feature maps
  Do: Reduce compute and increase receptive field
  Do Not: Over-pool tiny inputs

BatchNormalization:
  Use

## 1.1) Full Keras Layer Inventory (Auto-Detected from Your Environment)

This section lists all public layer classes available in tf.keras.layers for your installed TensorFlow version.
Use this as the complete reference list, while section 1 focuses on the most practical layers for system design.

In [3]:
# Import Python's inspect module to detect classes dynamically.
import inspect

# Collect all attributes from tf.keras.layers.
all_members = vars(layers)

# Keep only public classes that are subclasses of keras.layers.Layer.
all_layer_classes = []
for name, obj in all_members.items():
    # Skip private/internal names.
    if name.startswith('_'):
        continue
    # Keep only class objects.
    if not inspect.isclass(obj):
        continue
    # Keep only Keras Layer subclasses.
    if not issubclass(obj, keras.layers.Layer):
        continue
    # Store the class name.
    all_layer_classes.append(name)

# Sort layer names alphabetically for stable output.
all_layer_classes = sorted(set(all_layer_classes))

# Print summary count.
print(f'Total public Keras layer classes detected: {len(all_layer_classes)}')
# Print all layer names one per line.
for layer_name in all_layer_classes:
    print(layer_name)

Total public Keras layer classes detected: 165
Activation
ActivityRegularization
AdaptiveAveragePooling1D
AdaptiveAveragePooling2D
AdaptiveAveragePooling3D
AdaptiveMaxPooling1D
AdaptiveMaxPooling2D
AdaptiveMaxPooling3D
Add
AdditiveAttention
AlphaDropout
Attention
AugMix
AutoContrast
Average
AveragePooling1D
AveragePooling2D
AveragePooling3D
AvgPool1D
AvgPool2D
AvgPool3D
BatchNormalization
Bidirectional
CategoryEncoding
CenterCrop
Concatenate
Conv1D
Conv1DTranspose
Conv2D
Conv2DTranspose
Conv3D
Conv3DTranspose
ConvLSTM1D
ConvLSTM2D
ConvLSTM3D
Convolution1D
Convolution1DTranspose
Convolution2D
Convolution2DTranspose
Convolution3D
Convolution3DTranspose
Cropping1D
Cropping2D
Cropping3D
CutMix
Dense
DepthwiseConv1D
DepthwiseConv2D
Discretization
Dot
Dropout
ELU
EinsumDense
Embedding
Equalization
Flatten
FlaxLayer
GRU
GRUCell
GaussianDropout
GaussianNoise
GlobalAveragePooling1D
GlobalAveragePooling2D
GlobalAveragePooling3D
GlobalAvgPool1D
GlobalAvgPool2D
GlobalAvgPool3D
GlobalMaxPool1D
Glob

## 2) Functional Examples by Layer and When to Use

In [4]:
# Helper utility to compile and print summary for each example model.
def compile_and_show(model, title):
    # Compile with Adam optimizer for stable defaults.
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    # Print a visual separator between examples.
    print('\n' + '=' * 80)
    # Print the use-case title for context.
    print(title)
    # Print architecture details and parameter counts.
    model.summary()

# 2.1 Input + Dense for tabular baseline
# Define input contract: 16 numeric features per record.
tab_in = keras.Input(shape=(16,), name='tabular_features')
# First nonlinear projection layer.
x = layers.Dense(64, activation='relu')(tab_in)
# Second projection layer for representation refinement.
x = layers.Dense(32, activation='relu')(x)
# Binary output head: probability of churn.
tab_out = layers.Dense(1, activation='sigmoid', name='churn_prob')(x)
# Build end-to-end functional model graph.
tab_model = keras.Model(tab_in, tab_out, name='tabular_mlp')
# Compile and display summary for this architecture.
compile_and_show(tab_model, 'Input + Dense: use for tabular classification baseline')


Input + Dense: use for tabular classification baseline


Model: "tabular_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tabular_features (InputLayer)   │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ churn_prob (Dense)              │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,201 (12.50 KB)

 Trainable params: 3,201 (12.50 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# 2.2 Embedding + GRU for event or text sequence
# Input is a fixed-length sequence of integer token ids.
seq_in = keras.Input(shape=(50,), dtype='int32', name='token_ids')
# Map sparse token ids to dense vectors; mask_zero enables padding mask propagation.
e = layers.Embedding(input_dim=20000, output_dim=64, mask_zero=True)(seq_in)
# GRU encodes ordered dependencies into a single sequence representation.
e = layers.GRU(64)(e)
# Binary intent prediction head.
seq_out = layers.Dense(1, activation='sigmoid', name='intent_prob')(e)
# Build sequence model.
seq_model = keras.Model(seq_in, seq_out, name='embedding_gru_model')
# Compile and inspect architecture.
compile_and_show(seq_model, 'Embedding + GRU: use for ordered sequences with medium latency budget')


Embedding + GRU: use for ordered sequences with medium latency budget


Model: "embedding_gru_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 64)    │  1,280,000 │ token_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ token_ids[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 64)        │     24,960 │ embedding[0][0],  │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ intent_prob (Dense) │ (None, 1)         │         65 │ gru[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,305,025 (4.98 MB)

 Trainable params: 1,305,025 (4.98 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# 2.3 Conv2D + MaxPooling2D + GlobalAveragePooling2D for image classification
# Define image input contract: 128x128 RGB image.
img_in = keras.Input(shape=(128, 128, 3), name='image')
# Convolution learns local texture and edge features.
c = layers.Conv2D(32, 3, activation='relu', padding='same')(img_in)
# Batch norm improves optimization stability.
c = layers.BatchNormalization()(c)
# Downsample feature map to reduce compute and add translation robustness.
c = layers.MaxPooling2D()(c)
# Deeper convolution learns higher-level visual concepts.
c = layers.Conv2D(64, 3, activation='relu', padding='same')(c)
# Normalize again after deeper convolution.
c = layers.BatchNormalization()(c)
# Further spatial downsampling.
c = layers.MaxPooling2D()(c)
# Replace flatten with global pooling to keep parameter count small.
c = layers.GlobalAveragePooling2D()(c)
# Regularize dense head to reduce overfitting.
c = layers.Dropout(0.3)(c)
# Output defect probability.
img_out = layers.Dense(1, activation='sigmoid', name='defect_prob')(c)
# Build model graph.
img_model = keras.Model(img_in, img_out, name='conv_classifier')
# Compile and inspect architecture.
compile_and_show(img_model, 'Conv2D stack: use for visual local patterns and scalable inference')


Conv2D stack: use for visual local patterns and scalable inference


Model: "conv_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ defect_prob (Dense)             │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,841 (77.50 KB)

 Trainable params: 19,649 (76.75 KB)

 Non-trainable params: 192 (768.00 B)

### 2.3.1 Flatten Layer: When and Why

Flatten converts a tensor like `(batch, height, width, channels)` into `(batch, height * width * channels)`.
Use it when a Dense head needs a full vector input.
Prefer GlobalAveragePooling when feature maps are large and you want fewer parameters.

In [7]:
# Flatten layer demo: convert 3D feature map to 1D vector before Dense.
# Input represents a small feature map: 8x8 with 16 channels.
flat_in = keras.Input(shape=(8, 8, 16), name='feature_map')
# Flatten turns each sample from shape (8, 8, 16) to (1024,).
flat_vec = layers.Flatten(name='flatten_features')(flat_in)
# Dense head can now consume the vector.
flat_hidden = layers.Dense(64, activation='relu')(flat_vec)
# Binary output for demonstration.
flat_out = layers.Dense(1, activation='sigmoid', name='binary_target')(flat_hidden)
# Build model.
flatten_model = keras.Model(flat_in, flat_out, name='flatten_demo_model')
# Compile and inspect.
compile_and_show(flatten_model, 'Flatten layer: use to bridge spatial tensors to Dense layers')

# Optional shape check with dummy data.
dummy_feature_map = np.random.randn(2, 8, 8, 16).astype('float32')
# Predict to verify end-to-end execution.
dummy_pred = flatten_model.predict(dummy_feature_map, verbose=0)
print('Input shape:', dummy_feature_map.shape)
print('Output shape:', dummy_pred.shape)


Flatten layer: use to bridge spatial tensors to Dense layers


Model: "flatten_demo_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ feature_map (InputLayer)        │ (None, 8, 8, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_features (Flatten)      │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ binary_target (Dense)           │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,665 (256.50 KB)

 Trainable params: 65,665 (256.50 KB)

 Non-trainable params: 0 (0.00 B)

Input shape: (2, 8, 8, 16)
Output shape: (2, 1)


### 2.3.2 Flatten vs GlobalAveragePooling2D (Parameter and Latency View)

Assume a CNN feature map with shape `(H, W, C)` and a Dense head with `U` units.

Parameter formulas:
- With `Flatten -> Dense(U)`: params = `(H * W * C) * U + U`
- With `GlobalAveragePooling2D -> Dense(U)`: params = `C * U + U`

So the reduction factor is approximately `H * W` (for same `C` and `U`).

Latency guidance:
- Flatten path is often heavier in the dense head and memory bandwidth.
- GlobalAveragePooling2D path is usually faster and more stable for large maps.
- Keep Flatten when exact spatial details must be preserved for downstream Dense modeling.

In [8]:
# Flatten vs GlobalAveragePooling2D: side-by-side parameter math and model summary.

# Define comparison assumptions.
# Example feature map shape from a CNN backbone.
H, W, C = 16, 16, 64
# Dense head width (same for both designs to compare fairly).
U = 128

# Compute parameter counts for only the final Dense layer in each design.
# Flatten output size is H * W * C.
flatten_dense_params = (H * W * C) * U + U
# GAP output size is C.
gap_dense_params = C * U + U

# Compute reduction factor in dense parameters.
reduction_factor = flatten_dense_params / gap_dense_params

print('Assumed feature map: (H, W, C) =', (H, W, C))
print('Dense units U =', U)
print('Dense params with Flatten =', flatten_dense_params)
print('Dense params with GAP =', gap_dense_params)
print('Parameter reduction factor (Flatten/GAP) =', round(reduction_factor, 2), 'x')

# Build two tiny models to visualize architecture differences.
cmp_input = keras.Input(shape=(H, W, C), name='cmp_feature_map')

# Path A: Flatten then Dense.
a = layers.Flatten(name='cmp_flatten')(cmp_input)
a = layers.Dense(U, activation='relu', name='cmp_flatten_dense')(a)
a_out = layers.Dense(1, activation='sigmoid', name='cmp_flatten_out')(a)
flatten_cmp_model = keras.Model(cmp_input, a_out, name='flatten_path_model')

# Path B: GlobalAveragePooling2D then Dense.
b = layers.GlobalAveragePooling2D(name='cmp_gap')(cmp_input)
b = layers.Dense(U, activation='relu', name='cmp_gap_dense')(b)
b_out = layers.Dense(1, activation='sigmoid', name='cmp_gap_out')(b)
gap_cmp_model = keras.Model(cmp_input, b_out, name='gap_path_model')

# Compile both models for consistency.
flatten_cmp_model.compile(optimizer='adam', loss='binary_crossentropy')
gap_cmp_model.compile(optimizer='adam', loss='binary_crossentropy')

print('\n' + '=' * 80)
print('Model A: Flatten path')
flatten_cmp_model.summary()

print('\n' + '=' * 80)
print('Model B: GlobalAveragePooling2D path')
gap_cmp_model.summary()

Assumed feature map: (H, W, C) = (16, 16, 64)
Dense units U = 128
Dense params with Flatten = 2097280
Dense params with GAP = 8320
Parameter reduction factor (Flatten/GAP) = 252.08 x

Model A: Flatten path


Model: "flatten_path_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ cmp_feature_map (InputLayer)    │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cmp_flatten (Flatten)           │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cmp_flatten_dense (Dense)       │ (None, 128)            │     2,097,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cmp_flatten_out (Dense)         │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,097,409 (8.00 MB)

 Trainable params: 2,097,409 (8.00 MB)

...

Model: "gap_path_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ cmp_feature_map (InputLayer)    │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cmp_gap                         │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cmp_gap_dense (Dense)           │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cmp_gap_out (Dense)             │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,449 (33.00 KB)

 Trainable params: 8,449 (33.00 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# 2.4 MultiHeadAttention + LayerNormalization + residual Add
# Input already contains token embeddings: sequence length 40, embedding dim 64.
tok_in = keras.Input(shape=(40, 64), name='token_embeddings')
# Self-attention lets each token attend to all other tokens.
attn = layers.MultiHeadAttention(num_heads=4, key_dim=16)(tok_in, tok_in)
# Residual connection preserves original signal and improves gradient flow.
x = layers.Add()([tok_in, attn])
# Layer norm stabilizes activations for transformer blocks.
x = layers.LayerNormalization()(x)
# Feed-forward network expands channel dimension.
ffn = layers.Dense(128, activation='relu')(x)
# Project back to original embedding size for residual compatibility.
ffn = layers.Dense(64)(ffn)
# Second residual connection.
x = layers.Add()([x, ffn])
# Second normalization.
x = layers.LayerNormalization()(x)
# Pool sequence to one vector for classification.
x = layers.GlobalAveragePooling1D()(x)
# Binary classification head.
tr_out = layers.Dense(1, activation='sigmoid')(x)
# Build transformer-style block model.
transformer_block_model = keras.Model(tok_in, tr_out, name='simple_transformer_block')
# Compile and inspect architecture.
compile_and_show(transformer_block_model, 'Attention block: use when long range token interactions matter')


Attention block: use when long range token interactions matter


Model: "simple_transformer_block"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_embeddings    │ (None, 40, 64)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 40, 64)    │     16,640 │ token_embeddings… │
│ (MultiHeadAttentio… │                   │            │ token_embeddings… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 40, 64)    │          0 │ token_embeddings… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 40, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 40, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 40, 64)    │      8,256 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 40, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 40, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         65 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 33,537 (131.00 KB)

 Trainable params: 33,537 (131.00 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# 2.5 Multi-input system design pattern with Concatenate
# Numeric tabular inputs from business features.
num_in = keras.Input(shape=(12,), name='numeric_features')
# Single categorical id feature (example: country id).
cat_in = keras.Input(shape=(1,), dtype='int32', name='country_id')
# Tokenized query text input.
txt_in = keras.Input(shape=(30,), dtype='int32', name='query_tokens')

# Numeric branch: shallow MLP.
num_branch = layers.Dense(32, activation='relu')(num_in)
# Categorical branch: id embedding lookup.
cat_branch = layers.Embedding(input_dim=300, output_dim=8)(cat_in)
# Flatten 1x8 embedding to vector shape (8,).
cat_branch = layers.Flatten()(cat_branch)
# Text branch: token embedding with padding mask support.
txt_branch = layers.Embedding(input_dim=10000, output_dim=32, mask_zero=True)(txt_in)
# Sequence encoder for text signal.
txt_branch = layers.LSTM(32)(txt_branch)

# Fuse modalities into one joint representation.
joined = layers.Concatenate()([num_branch, cat_branch, txt_branch])
# Joint dense layer for feature interactions.
joined = layers.Dense(64, activation='relu')(joined)
# Regularize final shared representation.
joined = layers.Dropout(0.2)(joined)
# Binary conversion prediction head.
multi_out = layers.Dense(1, activation='sigmoid', name='conversion_prob')(joined)

# Build multi-input functional model.
multi_input_model = keras.Model(
    inputs=[num_in, cat_in, txt_in],
    outputs=multi_out,
    name='multi_modal_conversion_model'
)
# Compile and inspect architecture.
compile_and_show(multi_input_model, 'Concatenate: use for enterprise multi-source feature fusion')


Concatenate: use for enterprise multi-source feature fusion


Model: "multi_modal_conversion_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ country_id          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ query_tokens        │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_features    │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 8)      │      2,400 │ country_id[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 30, 32)    │    320,000 │ query_tokens[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 30)        │          0 │ query_tokens[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │        416 │ numeric_features… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 32)        │      8,320 │ embedding_2[0][0… │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 72)        │          0 │ dense_6[0][0],    │
│ (Concatenate)       │                   │            │ flatten[0][0],    │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      4,672 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conversion_prob     │ (None, 1)         │         65 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 335,873 (1.28 MB)

 Trainable params: 335,873 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

## 3) Special Edge Case Examples

In [11]:
# Edge Case 1: Variable-length padded sequences with masking
# Input shape (None,) means variable token length per sample.
edge_in = keras.Input(shape=(None,), dtype='int32', name='padded_tokens')
# mask_zero=True ensures token id 0 is treated as padding and ignored by downstream RNN.
x = layers.Embedding(input_dim=5000, output_dim=32, mask_zero=True)(edge_in)
# Bidirectional LSTM reads sequence in forward and backward directions.
x = layers.Bidirectional(layers.LSTM(16))(x)
# Binary output for demonstration.
edge_out = layers.Dense(1, activation='sigmoid')(x)
# Build masking-aware model.
mask_model = keras.Model(edge_in, edge_out, name='masking_model')
# Compile with basic binary loss.
mask_model.compile(optimizer='adam', loss='binary_crossentropy')

# Create padded sequences where 0 indicates pad token.
padded_batch = np.array([
    [7, 2, 9, 4, 0, 0],
    [5, 8, 1, 6, 3, 2],
    [4, 9, 0, 0, 0, 0]
])
# Dummy targets for one batch training step.
y_dummy = np.array([0, 1, 0])
# Run one train step to confirm graph and masking work.
mask_model.train_on_batch(padded_batch, y_dummy)
# Print outcome message.
print('Masking model handles padded variable lengths safely.')

Masking model handles padded variable lengths safely.


In [12]:
# Edge Case 2: BatchNormalization and Dropout behavior at inference time
# Define input with 10 features.
inp = keras.Input(shape=(10,))
# Dense hidden layer.
x = layers.Dense(16, activation='relu')(inp)
# BatchNormalization uses batch stats in training and moving averages in inference.
x = layers.BatchNormalization()(x)
# Dropout is active only during training mode.
x = layers.Dropout(0.5)(x)
# Binary output layer.
out = layers.Dense(1, activation='sigmoid')(x)
# Build model.
train_inf_model = keras.Model(inp, out)
# Compile for binary objective.
train_inf_model.compile(optimizer='adam', loss='binary_crossentropy')

# Generate random sample batch.
sample = np.random.randn(4, 10).astype('float32')
# Forward pass in training mode: dropout and batch stats are active.
pred_train_mode = train_inf_model(sample, training=True)
# Forward pass in inference mode: dropout off, moving averages for BN.
pred_infer_mode = train_inf_model(sample, training=False)
# Print mean prediction for training mode.
print('Training mode prediction mean:', float(tf.reduce_mean(pred_train_mode)))
# Print mean prediction for inference mode.
print('Inference mode prediction mean:', float(tf.reduce_mean(pred_infer_mode)))
# Explain why both means differ.
print('These differ because dropout and BN use different behavior by mode.')

Training mode prediction mean: 0.503402829170227
Inference mode prediction mean: 0.37732523679733276
These differ because dropout and BN use different behavior by mode.


In [13]:
# Edge Case 3: Class imbalance with output bias initialization
# Example counts for rare positive class.
positives = 100
# Example counts for dominant negative class.
negatives = 9900
# Initialize output bias to prior log-odds to stabilize early training.
initial_bias = np.log(positives / negatives)

# Define 20-feature tabular input.
imb_in = keras.Input(shape=(20,))
# Hidden dense layer.
x = layers.Dense(32, activation='relu')(imb_in)
# Output layer with bias initialized from class prior.
imb_out = layers.Dense(1, activation='sigmoid', bias_initializer=keras.initializers.Constant(initial_bias))(x)
# Build model.
imb_model = keras.Model(imb_in, imb_out)
# Compile with imbalance-sensitive monitoring metrics.
imb_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC', 'Precision', 'Recall'])

# Print bias value used for initialization.
print('Initial bias for imbalance:', round(float(initial_bias), 4))
# Explain where this pattern is useful.
print('Use this pattern in rare-event detection problems like fraud.')

Initial bias for imbalance: -4.5951
Use this pattern in rare-event detection problems like fraud.


## 4) Enterprise Examples with System Design

In [14]:
# Enterprise Example A: Recommender candidate ranker (Wide and Deep style)
# System design notes:
# - Online path: feature service -> model -> top-K ranking
# - Offline path: daily training with sampled negatives and calibrated threshold

# Dense user-level numerical features.
user_dense = keras.Input(shape=(10,), name='user_dense')
# Dense item-level numerical features.
item_dense = keras.Input(shape=(12,), name='item_dense')
# High-cardinality user id input.
user_id = keras.Input(shape=(1,), dtype='int32', name='user_id')
# High-cardinality item id input.
item_id = keras.Input(shape=(1,), dtype='int32', name='item_id')

# Learn dense representation for user ids.
u_id_vec = layers.Flatten()(layers.Embedding(2_000_000, 32)(user_id))
# Learn dense representation for item ids.
i_id_vec = layers.Flatten()(layers.Embedding(500_000, 32)(item_id))

# Wide component keeps direct memorization signal from dense features.
wide = layers.Concatenate()([user_dense, item_dense])
# Deep component learns nonlinear interactions across ids and dense features.
deep = layers.Concatenate()([u_id_vec, i_id_vec, user_dense, item_dense])
# First deep hidden layer.
deep = layers.Dense(128, activation='relu')(deep)
# Regularization for better generalization.
deep = layers.Dropout(0.2)(deep)
# Second deep hidden layer.
deep = layers.Dense(64, activation='relu')(deep)

# Merge wide and deep signals.
final = layers.Concatenate()([wide, deep])
# Final ranking probability output.
score = layers.Dense(1, activation='sigmoid', name='rank_score')(final)
# Build recommender ranker model.
ranker_model = keras.Model([user_dense, item_dense, user_id, item_id], score, name='enterprise_ranker')
# Compile with binary target and AUC monitoring.
ranker_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
# Print architecture summary.
ranker_model.summary()

Model: "enterprise_ranker"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_id             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_id             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 1, 32)     │ 64,000,000 │ user_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 1, 32)     │ 16,000,000 │ item_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_dense          │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_dense          │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 32)        │          0 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 86)        │          0 │ flatten_1[0][0],  │
│ (Concatenate)       │                   │            │ flatten_2[0][0],  │
│                     │                   │            │ user_dense[0][0], │
│                     │                   │            │ item_dense[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 128)       │     11,136 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 22)        │          0 │ user_dense[0][0], │
│ (Concatenate)       │                   │            │ item_dense[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 64)        │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 86)        │          0 │ concatenate_1[0]… │
│ (Concatenate)       │                   │            │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rank_score (Dense)  │ (None, 1)         │         87 │ concatenate_3[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 80,019,479 (305.25 MB)

 Trainable params: 80,019,479 (305.25 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# Enterprise Example B: Support ticket router (text + metadata)
# System design notes:
# - Inputs from ticketing API and CRM metadata
# - Model predicts queue and urgency in multi-task setup

# Tokenized support ticket text input.
ticket_tokens = keras.Input(shape=(80,), dtype='int32', name='ticket_tokens')
# Encoded priority code from ticket metadata.
priority_code = keras.Input(shape=(1,), dtype='int32', name='priority_code')
# Numeric account-level features from CRM.
account_features = keras.Input(shape=(6,), name='account_features')

# Convert tokens to vectors, then summarize sequence using bidirectional GRU.
txt = layers.Embedding(30000, 64, mask_zero=True)(ticket_tokens)
txt = layers.Bidirectional(layers.GRU(32))(txt)
# Small embedding branch for priority code.
pri = layers.Flatten()(layers.Embedding(10, 4)(priority_code))

# Fuse text, priority embedding, and account numeric features.
joint = layers.Concatenate()([txt, pri, account_features])
# Shared hidden representation for both tasks.
joint = layers.Dense(64, activation='relu')(joint)
# Stabilize representation before branching into heads.
joint = layers.LayerNormalization()(joint)

# Multi-class queue prediction head.
queue_head = layers.Dense(8, activation='softmax', name='queue_class')(joint)
# Binary urgency prediction head.
urgency_head = layers.Dense(1, activation='sigmoid', name='urgency_score')(joint)

# Build multi-task model with two outputs.
router_model = keras.Model(
    [ticket_tokens, priority_code, account_features],
    [queue_head, urgency_head],
    name='ticket_router_multitask'
)
# Compile with separate losses and metrics for each head.
router_model.compile(
    optimizer='adam',
    loss={'queue_class': 'sparse_categorical_crossentropy', 'urgency_score': 'binary_crossentropy'},
    metrics={'queue_class': ['accuracy'], 'urgency_score': ['AUC']}
)
# Print architecture summary.
router_model.summary()

Model: "ticket_router_multitask"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ticket_tokens       │ (None, 80)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ priority_code       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 80, 64)    │  1,920,000 │ ticket_tokens[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 80)        │          0 │ ticket_tokens[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 1, 4)      │         40 │ priority_code[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 64)        │     18,816 │ embedding_6[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 4)         │          0 │ embedding_7[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ account_features    │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 74)        │          0 │ bidirectional_1[… │
│ (Concatenate)       │                   │            │ flatten_3[0][0],  │
│                     │                   │            │ account_features… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 64)        │      4,800 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 64)        │        128 │ dense_15[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ queue_class (Dense) │ (None, 8)         │        520 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ urgency_score       │ (None, 1)         │         65 │ layer_normalizat… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,944,369 (7.42 MB)

 Trainable params: 1,944,369 (7.42 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Enterprise Example C: Fraud detection with temporal + static fusion
# System design notes:
# - Stream transactions in sequence and merge with profile risk features
# - Use threshold service to trade precision vs recall by business hour

# Sequence input: recent transactions represented by 20 features each.
txn_seq = keras.Input(shape=(30, 20), name='transaction_sequence')
# Static user/account profile features.
profile = keras.Input(shape=(15,), name='profile_features')

# Temporal encoder over transaction sequence.
seq_repr = layers.LSTM(64, return_sequences=True)(txn_seq)
# Self-attention to capture long-range interactions between transaction steps.
attn_repr = layers.MultiHeadAttention(num_heads=4, key_dim=16)(seq_repr, seq_repr)
# Residual connection between LSTM and attention output.
seq_repr = layers.Add()([seq_repr, attn_repr])
# Normalize temporal representation.
seq_repr = layers.LayerNormalization()(seq_repr)
# Pool sequence into fixed-size vector.
seq_repr = layers.GlobalAveragePooling1D()(seq_repr)

# Fuse temporal vector with static profile features.
fused = layers.Concatenate()([seq_repr, profile])
# Dense fusion layer.
fused = layers.Dense(128, activation='relu')(fused)
# Dropout regularization.
fused = layers.Dropout(0.3)(fused)
# Output fraud probability.
fraud_prob = layers.Dense(1, activation='sigmoid', name='fraud_prob')(fused)

# Build final fraud model.
fraud_model = keras.Model([txn_seq, profile], fraud_prob, name='fraud_detection_model')
# Compile with business-relevant metrics for imbalance.
fraud_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC', 'Precision', 'Recall'])
# Print architecture summary.
fraud_model.summary()

Model: "fraud_detection_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ transaction_sequen… │ (None, 30, 20)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 30, 64)    │     21,760 │ transaction_sequ… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 64)    │     16,640 │ lstm_2[0][0],     │
│ (MultiHeadAttentio… │                   │            │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 30, 64)    │          0 │ lstm_2[0][0],     │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ profile_features    │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 79)        │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ profile_features… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 128)       │     10,240 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 128)       │          0 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fraud_prob (Dense)  │ (None, 1)         │        129 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 48,897 (191.00 KB)

 Trainable params: 48,897 (191.00 KB)

 Non-trainable params: 0 (0.00 B)

## Final Notes

- Start with the simplest architecture that matches your latency and quality target.
- Add layers only when they add measurable business value.
- In enterprise systems, input contracts, feature lineage, and serving parity matter as much as model quality.
- Build with functional API when you have multiple inputs, branches, or multi-head outputs.